In [ ]:
import jax
import jax.numpy as jnp
import numpyro
import numpyro.distributions as dist
from numpyro import plate
from numpyro.distributions import constraints

In [ ]:
@jax.jit
def mix_weights(beta):
    # Compute cumulative product of (1 - beta) along the last dimension
    beta1m_cumprod = jnp.cumprod(1.0 - beta, axis=-1)

    # Pad beta with a 1 at the end of the last dimension
    beta_padded = jnp.pad(beta, ((0, 0),) * (beta.ndim - 1) + ((0, 1),), constant_values=1.0)

    # Pad beta1m_cumprod with a 1 at the start of the last dimension
    beta1m_cumprod_padded = jnp.pad(beta1m_cumprod, ((0, 0),) * (beta.ndim - 1) + ((1, 0),), constant_values=1.0)

    # Element-wise multiplication
    weight = beta_padded * beta1m_cumprod_padded

    # Numerical stability (avoid zero weights)
    weight = jnp.maximum(weight, 1e-6)

    # Normalize across last dimension
    rlt = weight / jnp.sum(weight, axis=-1, keepdims=True)

    return rlt

In [ ]:
def model(data, struct_upbd, vocab_size, device=None):
    """
    Args:
      data: (feature, label), where
            feature: (N, M, vocab_size) one-hot words per position (or None for prior)
            label:   (N,) regression targets (or None for prior)
      struct_upbd: dict like {"G0": K0, "G1": K1, ...} (top level is G0)
      vocab_size: int
      device: unused (kept for API parity)
    """
    param_dims = list(struct_upbd.values())
    param_dims.reverse()

    # ------------------------
    # Global/structural params
    # ------------------------
    struct_params = {}
    struct_params["gamma"]      = numpyro.param("model_gamma",      jnp.asarray([1.0]), constraint=constraints.positive)
    struct_params["nig_mu"]     = numpyro.param("model_nig_mu",     jnp.asarray([0.0]))
    struct_params["nig_kappa"]  = numpyro.param("model_nig_kappa",  jnp.asarray([1.0]), constraint=constraints.positive)
    struct_params["nig_alpha"]  = numpyro.param("model_nig_alpha",  jnp.asarray([1.0]), constraint=constraints.positive)
    struct_params["nig_beta"]   = numpyro.param("model_nig_beta",   jnp.asarray([1.0]), constraint=constraints.positive)

    # alpha/eta tensors across hierarchy
    for parent_level in range(len(struct_upbd) - 1):
        child_level = parent_level + 1
        full_dim = child_level + 1  

        base = numpyro.param(
            f"model_alpha{parent_level}",
            jnp.ones(tuple(param_dims[-child_level:-1])),
            constraint=constraints.positive,
        )
        struct_params[f"alpha{parent_level}"] = jnp.expand_dims(base, -1) * jnp.ones(tuple(param_dims[-child_level:]))
        assert struct_params[f"alpha{parent_level}"].shape == tuple(param_dims[-child_level:])

        struct_params[f"eta{parent_level}"] = numpyro.param(
            f"model_eta{parent_level}",
            jnp.ones(tuple(param_dims[-child_level:-1])),
            constraint=constraints.positive,
        )
        assert struct_params[f"eta{parent_level}"].shape == tuple(param_dims[-child_level:-1])

    # Last level alpha (no eta)
    last_idx = len(struct_upbd) - 1
    base_last = numpyro.param(
        f"model_alpha{last_idx}",
        jnp.ones(tuple(param_dims[:-1])),
        constraint=constraints.positive,
    )
    struct_params[f"alpha{last_idx}"] = jnp.expand_dims(base_last, -1) * jnp.ones(tuple(param_dims))

    # ---------------
    # Stick-breaking
    # ---------------
    struct_values = {}

    # Top-level Beta sticks B0 -> G0 weights
    K0 = param_dims[-1]
    B0_a = jnp.ones((K0,))
    B0_b = jnp.broadcast_to(struct_params["gamma"], (K0,))
    beta_0 = dist.Beta(B0_a, B0_b).sample()
    struct_values["P0"] = (B0_a, B0_b)
    struct_values["B0"] = beta_0
    struct_values["G0"] = mix_weights(beta_0)[..., :-1]  # (K0,)

    # Lower levels
    for parent_level in range(len(struct_upbd) - 1):
        child_level = parent_level + 1
        full_dim = child_level + 1  # number of dims for this plate

        # shapes like in your code:
        # alpha * G_parent and alpha * (1 - cumsum(G_parent))
        G_parent = struct_values[f"G{parent_level}"]  # shape param_dims[-(parent_level+1):]
        alpha_param = struct_params[f"alpha{parent_level}"]  # shape param_dims[-child_level:]
        param_alpha = alpha_param * G_parent
        param_beta = alpha_param * (1.0 - jnp.cumsum(G_parent, axis=-1))

        # Expand to include one extra leading dim as in the PyTorch code
        shape_needed = tuple(param_dims[-full_dim:])
        a = jnp.broadcast_to(jnp.expand_dims(param_alpha, 0), shape_needed)
        b = jnp.broadcast_to(jnp.expand_dims(param_beta, 0), shape_needed)

        beta = dist.Beta(a, b).sample()
        struct_values[f"P{child_level}"] = (a, b)
        struct_values[f"B{child_level}"] = beta
        struct_values[f"G{child_level}"] = mix_weights(beta)[..., :-1]
        assert struct_values[f"G{child_level}"].shape == tuple(param_dims[-full_dim:])

    # ---------------
    # Cluster weights
    # ---------------
    for parent_level in range(len(struct_upbd) - 1):
        child_level = parent_level + 1
        full_dim = child_level + 1
        eta = struct_params[f"eta{parent_level}"]  # shape param_dims[-full_dim:-1]
        ones_like = jnp.ones_like(eta)

        a = jnp.broadcast_to(jnp.expand_dims(ones_like, 0), tuple(param_dims[-full_dim:-1]))
        b = jnp.broadcast_to(jnp.expand_dims(eta, 0), tuple(param_dims[-full_dim:-1]))

        beta = dist.Beta(a, b).to_event(child_level).sample()
        struct_values[f"L{parent_level}"] = mix_weights(beta)[..., :-1]  # categorical probs over next level

    # -----------------------
    # Mixture components
    # -----------------------
    # Topics over vocab
    G0_size = struct_upbd["G0"]
    mixture_components = {}
    mixture_components["generation"] = dist.Dirichlet(
            jnp.broadcast_to(struct_params["gamma"], (G0_size,))[:, None]
            * jnp.ones((G0_size, vocab_size))
        ).sample()
    # Regression components via NIG prior
    sigma = dist.InverseGamma(
            jnp.broadcast_to(struct_params["nig_alpha"], (G0_size,)),
            jnp.broadcast_to(struct_params["nig_beta"],  (G0_size,))
        ).sample()
    mu = dist.Normal(
            jnp.broadcast_to(struct_params["nig_mu"], (G0_size,)),
            jnp.sqrt(sigma / jnp.broadcast_to(struct_params["nig_kappa"], (G0_size,)))
        ).sample(1)
    mixture_components["regression_sigma"] = sigma
    mixture_components["regression_mu"] = mu

    # -----------------------
    # Data handling
    # -----------------------
    feature = data[0]          # expected shape (N, M, vocab_size)
    N = feature.shape[0]
    M = feature.shape[1]
    label = data[1] 

    # ---------------------------------
    # Per-document hierarchical routing
    # ---------------------------------
    assigned_zs = [jnp.zeros((N,), dtype=jnp.int32)]  # seed like your torch.zeros long
    doc_values = {}

    with plate("Data", N):
        # Walk down the cluster tree
        for level in range(len(struct_upbd) - 1):
            # cluster_weights[L{level}] has dims over the parent categories, last dim = choices at this level
            W = struct_values[f"L{level}"]  # shape e.g. (..., K_level)
            # Build advanced indexing tuple from already-sampled parent zs
            # assigned_zs holds arrays of shape (N,), one per parent axis.
            index_tuple = tuple(assigned_zs[:]) + (slice(None),)
            # param -> (N, K_level)
            param = W[index_tuple]
            z = dist.Categorical(probs=param).sample()
            assigned_zs.append(z)

        # reverse to align with your later indexing usage
        assigned_zs = assigned_zs[::-1]

        # Document-level stick-breaking at bottom: construct Beta params using G_{L} and alpha_{L}
        bottom_G = struct_values[f"G{len(struct_upbd)-1}"]  # shape (K0,) broadcastable with per-doc gather
        bottom_alpha = struct_params[f"alpha{len(struct_upbd)-1}"]

        # Gather per-doc parent path for alpha and weights_prior
        # For G_{L} (topic base weights), use parent indices in assigned_zs[:-1]
        idx_tuple_weights = tuple(assigned_zs[:-1]) + (slice(None),)
        weights_prior = bottom_G[idx_tuple_weights]  # (N, G0)
        concentrate = bottom_alpha[idx_tuple_weights[:-1]]  # (N,)

        param_alpha = concentrate[..., None] * weights_prior
        param_beta = concentrate[..., None] * (1.0 - jnp.cumsum(weights_prior, axis=-1))

        beta_doc = dist.Beta(param_alpha, param_beta).sample()  # (N, G0)
        doc_values["P"] = [param_alpha, param_beta]
        doc_values["B"]  = beta_doc

        topic_dist = mix_weights(beta_doc)[..., :-1]   # (N, G0)
        doc_values["G"] = topic_dist

        # ----- Words -----
        # Per-token topic → per-token word dist, then one-hot word obs via Multinomial(1, .)
        topic_over_docs = jnp.broadcast_to(topic_dist[:, None, :], (N, M, G0_size))
        z_gen = dist.Categorical(probs=topic_over_docs).sample()  # (N, M)
        obs = feature

        # ----- Regression -----
        z_reg = dist.Categorical(probs=topic_dist).sample()  
        reg = label

    # Recreate category_assignments like your torch.stack(cat_zs, dim=1)
    cat_zs = assigned_zs[:-1][::-1]  # drop the seed, reverse to original order
    category_assignments = jnp.stack(cat_zs, axis=1) if len(cat_zs) > 0 else jnp.zeros((N, 0), dtype=jnp.int32)

    return {
        "struct_params": struct_params,
        "struct_values": struct_values,
        "mixture_components": mixture_components,
        "category_assignments": category_assignments,
        "doc_values": doc_values,
        "words": {
            "z_gen": z_gen,
            "z_reg": z_reg,
            "obs": obs,
            "reg": reg,
        },
    }

In [ ]:
@jax.jit
def suffix_sum(x: jnp.ndarray) -> jnp.ndarray:
    """
    Compute suffix sums along the last dimension of a tensor.
    Each entry is the sum of all elements to its right.
    The last element along that dimension is always 0.
    
    Example:
        x = jnp.array([1,2,3])
        suffix_sum(x) -> [5,3,0]
        
        x = jnp.array([[1,2,3],[4,5,6]])
        suffix_sum(x) -> [[5,3,0],
                           [11,6,0]]
    """
    # Flip along the last dimension
    rev = jnp.flip(x, axis=-1)
    # Cumulative sum on the flipped tensor
    rev_cumsum = jnp.cumsum(rev, axis=-1)
    # Flip back
    suffix = jnp.flip(rev_cumsum, axis=-1)
    # Subtract original to exclude current element
    suffix = suffix - x
    return suffix

In [ ]:
@jax.jit
def word_category_conditional(word, weight, components):
    gen_dist = dist.Multinomial(total_count=1, probs=components)
    log_probs = gen_dist.log_prob(word)
    un_normalized = log_probs + jnp.log(weight + 1e-10)
    cat_prob = un_normalized/un_normalized.sum(axis=-1, keepdims=True)
    sample = dist.Categorical(probs=cat_prob).sample()
    return sample

In [ ]:
@jax.jit
def reg_category_conditional(score, weight, components):
    reg_dist = dist.Normal(loc=components[0], scale=jnp.sqrt(components[1]))
    log_probs = reg_dist.log_prob(score)
    un_normalized = log_probs + jnp.log(weight + 1e-10)
    cat_prob = un_normalized/un_normalized.sum(axis=-1, keepdims=True)
    sample = dist.Categorical(probs=cat_prob).sample()
    return sample

In [ ]:
@jax.jit
def doc_categories_conditional(cats, nu_doc, nu1, nu2, params0, params1, params2, cluster_prob0, cluster_prob1, S):
    nu_1_log_prob = dist.Beta(params0[0], params0[1]).log_prob(nu1)
    nu_2_log_prob = dist.Beta(params1[0], params1[1]).log_prob(nu2)
    mu_doc_log_prob = dist.Beta(params2[0], params2[1]).log_prob(nu_doc)
    cat_log_prob = jnp.log(cluster_prob0[cats[0]]) + jnp.log(cluster_prob1[cats[1]])
    raw_prob = jnp.exp(nu_1_log_prob + nu_2_log_prob + mu_doc_log_prob + cat_log_prob)
    prob = raw_prob / jnp.sum(raw_prob)
    sample = dist.Categorical(probs=prob).sample()
    new_cat0 = sample // S
    new_cat1 = sample % S
    new_cat = jnp.array([new_cat0, new_cat1])
    return new_cat

In [ ]:
@jax.jit
def doc_weight_conditional(nu_doc, params, word_cats):
    cat_idx, cat_count = jnp.unique(word_cats, return_counts=True)
    alpha_bias = jnp.zeros_like(nu_doc, dtype=jnp.int32)
    alpha_bias = alpha_bias.at[cat_idx].set(cat_count)
    beta_bias = suffix_sum(alpha_bias)
    new_params = (params[0] + alpha_bias, params[1] + beta_bias)
    return new_params

In [ ]:
@jax.jit
def cat_weight_conditional(nu, params, word_cats):
    cat_idx, cat_count = jnp.unique(word_cats, return_counts=True)
    alpha_bias = jnp.zeros_like(nu, dtype=jnp.int32)
    alpha_bias = alpha_bias.at[cat_idx].set(cat_count)
    beta_bias = suffix_sum(alpha_bias)
    new_params = (params[0] + alpha_bias, params[1] + beta_bias)
    return new_params

In [ ]:
@jax.jit
def reg_component_conditional(obs, params):
    count = float(obs.size)
    mean = jnp.mean(obs)
    sum_var = jnp.sum((obs - mean)**2, keepdims=True)
    kappa = params[1] + count
    mu = (params[1]*params[0] + count*mean) / kappa
    alpha = params[2] + count / 2
    beta = params[3] + 0.5 * sum_var + (params[1] * count * (mean - params[0])**2) / (2 * kappa)
    new_params = (mu, kappa, alpha, beta)
    new_sigma = dist.InverseGamma(alpha, beta).sample()
    new_mu = dist.Normal(mu, jnp.sqrt(new_sigma / kappa)).sample()
    normal_params = (new_mu, new_sigma)
    return normal_params

In [ ]:
@jax.jit
def gen_component_conditional(obs, params):
    value = jnp.sum(obs, axis=-1)
    new_params = params + value
    sample = dist.Dirichlet(new_params).sample()
    return sample

In [ ]:
def gibbs_sampler(rng_key, state, struct_upbd, vocab_size, num_iters):
    """
    A simple Gibbs sampler for the HDMM model.
    
    Args:
      rng_key: JAX random key

    """
    struct_params = state["struct_params"]
    struct_values = state["struct_values"]
    generation_components = state["mixture_components"]["generation"]
    regression_mu = state["mixture_components"]["regression_mu"]
    regression_sigma = state["mixture_components"]["regression_sigma"]
    category_assignments = state["category_assignments"]
    doc_values = state["doc_values"]
    words = state["words"]
    z_gen = words["z_gen"]
    z_reg = words["z_reg"]
    obs = words["obs"]
    reg = words["reg"]

    # gibbs iterations
    for it in range(num_iters):
        # ------------------------
        # Sample mixture components
        # ------------------------

        # Topics over vocab
        G0_size = struct_upbd["G0"]
        for k in range(G0_size):
            word_idx = (z_gen == k)
            if jnp.sum(word_idx) > 0:
                obs_k = obs[word_idx]
                generation_components.at[k].set(
                    gen_component_conditional(obs_k, struct_params["gamma"])
                )
        # Regression components via NIG prior
        for k in range(G0_size):
            reg_idx = (z_reg == k)
            if jnp.sum(reg_idx) > 0:
                reg_k = reg[reg_idx]
                regression_mu.at[k].set(
                    reg_component_conditional(reg_k, (
                        struct_params["nig_mu"],
                        struct_params["nig_kappa"],
                        struct_params["nig_alpha"],
                        struct_params["nig_beta"]
                    ))[0]
                )
                regression_sigma.at[k].set(
                    reg_component_conditional(reg_k, (
                        struct_params["nig_mu"],
                        struct_params["nig_kappa"],
                        struct_params["nig_alpha"],
                        struct_params["nig_beta"]
                    ))[1]
                )

        # ------------------------
        # Sample category assignments
        # ------------------------
        N = obs.shape[0]
        new_cat_assignments = []
        for n in range(N):
            new_cat = doc_categories_conditional(
                category_assignments[n],
                doc_values["G"][n],
                struct_values["B1"][category_assignments[n, 0]],
                struct_values["B2"][category_assignments[n, 1], category_assignments[n, 0]],
                struct_values["P0"][category_assignments[n, 0]],
                struct_values["P1"][category_assignments[n, 1], category_assignments[n, 0]],
                struct_values["P2"][category_assignments[n, 2], category_assignments[n, 1]],
                struct_values["L0"],
                struct_values["L1"][category_assignments[n, 0]],
                struct_values["L2"][category_assignments[n, 2], category_assignments[n, 1]]
            )
            new_cat_assignments.append(new_cat)

            # Update doc-level weights after changing category assignment
            doc_values["P"] = doc_weight_conditional(
                doc_values["B"][n],
                struct_values[f"P{len(struct_upbd)-1}"][category_assignments[n][1], category_assignments[n][0]],
                z_gen[n]
            )
            doc_values["B"] = dist.Beta(doc_values["P"][0], doc_values["P"][1]).sample(rng_key)
            doc_values["G"] = mix_weights(doc_values["B"])[..., :-1]

            # Update word and regression category assignments
            for m in range(obs.shape[1]):
                z_gen = word_category_conditional(
                    obs[n, m],
                    doc_values["G"][n],
                    generation_components
                )
            z_reg = reg_category_conditional(
                reg[n],
                doc_values["G"][n],
                (regression_mu, regression_sigma)
            )

        # Update category assignments
        category_assignments = jnp.array(new_cat_assignments)

        # ------------------------
        # Sample document weights
        # ------------------------
        for n in range(N):
            new_doc_params = doc_weight_conditional(
                doc_values["B"][n],
                struct_values[f"P{len(struct_upbd)-1}"][category_assignments[n][1], category_assignments[n][0]],
                z_gen[n]
            )
            doc_values["P"] = new_doc_params
            doc_values["B"] = dist.Beta(new_doc_params[0], new_doc_params[1]).sample(rng_key)
            doc_values["G"] = mix_weights(doc_values["B"])[..., :-1]

        # ------------------------
        # Struct variables
        # ------------------------
        for s in range(struct_upbd["G1"]):
            mask = (category_assignments[:, 1] == s)

            row_idx = jnp.where(mask)[0]
            struct_values["P1"][s] = cat_weight_conditional(
                struct_values["B1"][s],
                struct_values["P0"],
                z_gen[row_idx]
            )
            struct_values["B1"][s] = dist.Beta(struct_values["P1"][s][0], struct_values["P1"][s][1]).sample(rng_key)
            struct_values["G1"][s] = mix_weights(struct_values["B1"])[..., :-1]
            for c in range(struct_upbd["G2"]):
                mask = (category_assignments[:, 2] == c) & (category_assignments[:, 1] == s)
                row_idx = jnp.where(mask)[0]
                struct_values["P2"][c, s] = cat_weight_conditional(
                    struct_values["B2"][c, s],
                    struct_values["P1"][s],
                    z_gen[row_idx]
                )
                struct_values["B2"][c, s] = dist.Beta(struct_values["P2"][c, s][0], struct_values["P2"][c, s][1]).sample(rng_key)
                struct_values["G2"][c, s] = mix_weights(struct_values["B2"])[..., :-1]


    post_state = {
        "struct_params": struct_params,
        "struct_values": struct_values,
        "mixture_components": {
            "generation": generation_components,
            "regression_mu": regression_mu,
            "regression_sigma": regression_sigma,
        },
        "category_assignments": category_assignments,
        "doc_values": doc_values,
        "words": {
            "z_gen": z_gen,
            "z_reg": z_reg,
            "obs": obs,
            "reg": reg,
        },
    }
    return post_state